In [ ]:
# ============================================================
# NOTEBOOK : Nettoyage et Pré-traitement — UAV-NIDD
# Dataset  : UAV Network Intrusion Detection Dataset
# ============================================================

# ────────────────────────────────────────
# 1. IMPORTATION DES LIBRAIRIES
# ────────────────────────────────────────
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.preprocessing import StandardScaler, LabelEncoder
import os

# ────────────────────────────────────────
# 2. CHARGEMENT DU DATASET
# ────────────────────────────────────────
UAV_NIDD = r"C:\Drone_Attack_Similarity_Project\DATASET\UAV_NIDD\Normal\Normal.csv"
df = pd.read_csv(UAV_NIDD)
df_original = df.copy()

print(f" Dataset chargé : {df.shape[0]:,} lignes × {df.shape[1]} colonnes")


# ────────────────────────────────────────
# 3. SUPPRESSION DES DOUBLONS
# ────────────────────────────────────────
nb_doublons = df.duplicated().sum()
print(f" Doublons détectés : {nb_doublons}")

if nb_doublons > 0:
    df = df.drop_duplicates()
    print(f" Doublons supprimés — nouveau shape : {df.shape}")
else:
    print(" Aucun doublon détecté")


# ────────────────────────────────────────
# 4. GESTION DES VALEURS MANQUANTES
# ────────────────────────────────────────
missing = df.isnull().sum()
total_missing = missing.sum()

print(f" Total valeurs manquantes : {total_missing}\n")

if total_missing > 0:
    num_cols = df.select_dtypes(include=[np.number]).columns
    cat_cols = df.select_dtypes(include=['object']).columns

    # Colonnes numériques → médiane
    for col in num_cols:
        if df[col].isnull().sum() > 0:
            median_val = df[col].median()
            df[col].fillna(median_val, inplace=True)
            print(f" {col} — rempli par médiane ({median_val:.4f})")

    # Colonnes catégorielles → mode
    for col in cat_cols:
        if df[col].isnull().sum() > 0:
            mode_val = df[col].mode()[0]
            df[col].fillna(mode_val, inplace=True)
            print(f"  {col} — rempli par mode ({mode_val})")
else:
    print(" Aucune valeur manquante")


# ────────────────────────────────────────
# 5. SUPPRESSION DES COLONNES INUTILES
# ────────────────────────────────────────
# Détection automatique des colonnes à supprimer :
# colonnes avec variance nulle ou identifiant unique
cols_to_drop = []

for col in df.columns:
    if col == 'label':
        continue
    # Colonne avec une seule valeur unique
    if df[col].nunique() == 1:
        cols_to_drop.append(col)
    # Colonne identifiant (autant de valeurs uniques que de lignes)
    if df[col].nunique() == len(df):
        cols_to_drop.append(col)

cols_to_drop = list(set(cols_to_drop))

if cols_to_drop:
    df = df.drop(columns=cols_to_drop)
    print(f" Colonnes supprimées : {cols_to_drop}")
else:
    print(" Aucune colonne inutile détectée")

print(f"   Shape après suppression : {df.shape}")



# ────────────────────────────────────────
# 6. TRAITEMENT DES OUTLIERS (IQR)
# ────────────────────────────────────────
num_cols = df.select_dtypes(include=[np.number]).columns.tolist()

print(" Traitement des outliers par écrêtage (capping IQR) :\n")

nb_total_outliers = 0
for col in num_cols:
    Q1  = df[col].quantile(0.25)
    Q3  = df[col].quantile(0.75)
    IQR = Q3 - Q1
    lower = Q1 - 1.5 * IQR
    upper = Q3 + 1.5 * IQR

    nb_outliers = ((df[col] < lower) | (df[col] > upper)).sum()
    nb_total_outliers += nb_outliers

    if nb_outliers > 0:
        df[col] = df[col].clip(lower=lower, upper=upper)
        print(f" {col} — {nb_outliers} outliers écrêtés")

print(f"\n Total outliers traités : {nb_total_outliers}")


# ────────────────────────────────────────
# 7. NORMALISATION DES FEATURES (StandardScaler)
# ────────────────────────────────────────
num_cols = df.select_dtypes(include=[np.number]).columns.tolist()

scaler = StandardScaler()
df[num_cols] = scaler.fit_transform(df[num_cols])

print(f" Normalisation appliquée sur {len(num_cols)} features")
print(f" Méthode : StandardScaler (moyenne=0, écart-type=1)")
print(f"\n   Vérification après normalisation :")
print(df[num_cols].describe().round(2).T[['mean', 'std', 'min', 'max']])




# ────────────────────────────────────────
# 8. ENCODAGE DU LABEL
# ────────────────────────────────────────

# Détection automatique de la colonne label
label_candidates = ['label', 'Label', 'attack_type',
                    'Attack', 'class', 'Class', 'target']
label_col = None
for col in label_candidates:
    if col in df.columns:
        label_col = col
        break

if label_col:
    print(f" Colonne label détectée : '{label_col}'")
    print(f"\n   Classes avant encodage :")
    print(df[label_col].value_counts())

    le = LabelEncoder()
    df['label_encoded'] = le.fit_transform(df[label_col])

    print(f"\n Encodage appliqué :")
    mapping = dict(zip(le.classes_, le.transform(le.classes_)))
    for classe, code in mapping.items():
        print(f"   {classe} → {code}")
else:
    print("    Aucune colonne label trouvée")
    print(f"   Colonnes disponibles : {df.columns.tolist()}")


# ────────────────────────────────────────
# 9. VÉRIFICATION FINALE
# ────────────────────────────────────────
print(" Vérification finale :\n")
print(f"   Shape original           : {df_original.shape}")
print(f"   Shape nettoyé            : {df.shape}")
print(f"   Valeurs manquantes       : {df.isnull().sum().sum()}")
print(f"   Doublons restants        : {df.duplicated().sum()}")

df.head()


# ────────────────────────────────────────
# 10. SAUVEGARDE DU DATASET NETTOYÉ
# ────────────────────────────────────────
OUTPUT_PATH = r"C:\Drone_Attack_Similarity_Project\Rapport\tables\UAV-NIDD_clean.csv"


df.to_csv(OUTPUT_PATH, index=False)
print(f"   Dataset nettoyé sauvegardé : {OUTPUT_PATH}")
print(f"   Shape final : {df.shape[0]:,} lignes × {df.shape[1]} colonnes")

 Dataset chargé : 149,434 lignes × 45 colonnes
 Doublons détectés : 112628
 Doublons supprimés — nouveau shape : (36806, 45)
 Total valeurs manquantes : 0

 Aucune valeur manquante
 Colonnes supprimées : ['fwd_seg_size_min']
   Shape après suppression : (36806, 44)
 Traitement des outliers par écrêtage (capping IQR) :

 flow_duration — 8364 outliers écrêtés
 flow_byts_s — 7976 outliers écrêtés
 flow_pkts_s — 7982 outliers écrêtés
 fwd_pkts_s — 8156 outliers écrêtés
 bwd_pkts_s — 8343 outliers écrêtés
 tot_fwd_pkts — 4727 outliers écrêtés
 tot_bwd_pkts — 4768 outliers écrêtés
 totlen_fwd_pkts — 3869 outliers écrêtés
 totlen_bwd_pkts — 8876 outliers écrêtés
 fwd_pkt_len_max — 432 outliers écrêtés
 fwd_pkt_len_min — 393 outliers écrêtés
 fwd_pkt_len_mean — 342 outliers écrêtés
 fwd_pkt_len_std — 5440 outliers écrêtés
 bwd_pkt_len_max — 5886 outliers écrêtés
 bwd_pkt_len_min — 1004 outliers écrêtés
 bwd_pkt_len_mean — 5886 outliers écrêtés
 bwd_pkt_len_std — 4985 outliers écrêtés
 pkt_len_